# 第5章 金融数据与时间边界

> **核心问题**：金融数据为什么不能“下载后直接建模”？一条数值在什么时候产生、什么时候发布、什么时候可交易，往往比数值本身更重要。

- 金融线：行情、成交量、公司行动、财务与宏观数据、复权和可知时间。
- 数学线：采样频率、缺失、对齐和信息集合。
- Python线：DatetimeIndex、审计、去重、缺失处理、重采样、`merge_asof`和数据血缘。

## AI学习状态

当前进度：第5章开始  
已掌握：股票、指数、交易和表格基础  
仍然薄弱：待填写  
下一步：清洗前先保留原始数据并生成审计报告。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 5.1 常见金融数据不只有收盘价

- 行情：开、高、低、收、成交量、买卖报价；
- 公司行动：分红、拆股、配股、停牌和退市；
- 基本面：财务报表及其公告时间；
- 宏观：统计期、发布日期、修订版本；
- 另类数据：新闻、文本、网络或卫星数据及其授权和时间戳。

每个数据集至少需要回答：对象是谁、字段含义、单位、频率、时区、来源、许可、发布时间和修订规则。

## 5.2 金融数据侦探：先检查一份故意损坏的数据

下表包含乱序日期、重复行、缺失值、异常成交量和无法解析的日期。不要一上来就`dropna()`。

In [ ]:
raw = pd.DataFrame({
    "date": ["2026-01-05", "2026-01-02", "2026-01-06", "2026-01-06", "bad-date", "2026-01-08"],
    "close": [10.20, 10.00, np.nan, 10.30, 10.50, 10.40],
    "volume": [1200, 1000, 1500, 1500, -20, 1800],
    "source": ["demo"] * 6,
})
raw

In [ ]:
def audit_frame(df, date_col="date"):
    parsed = pd.to_datetime(df[date_col], errors="coerce")
    return {
        "rows": len(df),
        "unparseable_dates": int(parsed.isna().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_dates": int(parsed.duplicated(keep=False).sum()),
        "missing_by_column": df.isna().sum().to_dict(),
        "negative_volume": int((df.get("volume", pd.Series(dtype=float)) < 0).sum()),
        "is_date_sorted": bool(parsed.dropna().is_monotonic_increasing),
    }


audit_frame(raw)

**Python提示：`errors="coerce"`**

无法解析的日期会变成`NaT`，便于统计和定位。它不会自动证明这些行可以删除；清洗决定必须结合来源重新核查。

**量化编程警告**：重复日期不一定等于重复记录。不同交易所、资产、报价类型或日内时点可能共享日期，必须先确定唯一键。

In [ ]:
cleaned = raw.copy()
cleaned["date"] = pd.to_datetime(cleaned["date"], errors="coerce")
cleaned = cleaned.dropna(subset=["date"])
cleaned = cleaned[cleaned["volume"] >= 0]
cleaned = cleaned.drop_duplicates(subset=["date"], keep="last")
cleaned = cleaned.sort_values("date").set_index("date")
cleaned

### 清洗决策记录

我们选择保留重复日期的最后一条，但这只是演示规则。真实项目必须说明为什么“最后一条”更可信，并保存被排除记录。

### 我的审计意见

哪些问题可以自动处理，哪些必须返回数据源核实？为什么缺失收盘价不能一律前向填充？

<!-- 在这里填写；完成前AI不要代答 -->

## 5.3 交易日缺失与数据缺失不是一回事

周末、节假日和停牌可能没有交易；接口失败也可能造成缺失。把自然日强行补齐并前向填充，会创造并不存在的成交记录。

In [ ]:
business_days = pd.date_range(cleaned.index.min(), cleaned.index.max(), freq="B")
reindexed = cleaned.reindex(business_days)
reindexed.index.name = "date"
reindexed

In [ ]:
fig, ax = plt.subplots()
ax.plot(reindexed.index, reindexed["close"], "o-", label="原始可用收盘价")
ax.plot(reindexed.index, reindexed["close"].ffill(), "x--", label="前向填充（仅演示）")
ax.set(title="填充值不是新观察", xlabel="日期", ylabel="价格")
ax.legend(); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

## 5.4 公司行动与复权：价格跳变不一定是亏损

假设一股拆成两股，拆股前每股100元，拆股后理论价格约50元。只看未复权价格会显示约-50%，但持股数翻倍，财富未因此减半。

In [ ]:
split_data = pd.DataFrame({
    "date": pd.date_range("2026-02-02", periods=6, freq="B"),
    "raw_close": [96, 98, 100, 50, 51, 52],
    "shares_held": [10, 10, 10, 20, 20, 20],
}).set_index("date")
split_data["position_value"] = split_data["raw_close"] * split_data["shares_held"]
split_data["naive_return"] = split_data["raw_close"].pct_change()
split_data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
split_data["raw_close"].plot(ax=axes[0], marker="o", title="未复权每股价格")
split_data["position_value"].plot(ax=axes[1], marker="o", title="持仓价值（股数已调整）")
axes[0].set_ylabel("元/股"); axes[1].set_ylabel("元"); plt.tight_layout(); plt.show()

**量化编程警告：复权方法取决于研究目的**。研究交易成交要保留当时可交易价格；研究总回报要正确纳入分红和股数变化。不能只看到接口中的`adjusted close`就假定定义一致。

## 5.5 频率转换：日数据到月数据

对价格常取期末值，对成交量常求和，对收益率则应根据定义复合。`resample`不会替你选择金融上正确的聚合规则。

In [ ]:
rng = np.random.default_rng(5)
dates = pd.date_range("2025-01-01", periods=120, freq="B")
daily = pd.DataFrame({
    "close": 100 * np.cumprod(1 + rng.normal(0.0003, 0.01, len(dates))),
    "volume": rng.integers(1_000, 5_000, len(dates)),
}, index=dates)
monthly = daily.resample("ME").agg({"close": "last", "volume": "sum"})
monthly["return"] = monthly["close"].pct_change()
monthly.head()

## 5.6 最危险的错误：把发布日期之前的数据用于决策

宏观数据有“统计期”和“发布日期”。一季度数据可能在4月发布；模型不能从1月起就使用最终公布值。财报、指数成分和修订数据同理。

In [ ]:
market = pd.DataFrame({
    "time": pd.date_range("2026-04-01 09:00", periods=8, freq="h"),
    "price": [100, 101, 100.5, 101.5, 102, 101.8, 102.4, 102.1],
})
releases = pd.DataFrame({
    "release_time": pd.to_datetime(["2026-04-01 11:30", "2026-04-01 15:30"]),
    "indicator": [4.8, 5.1],
})

aligned = pd.merge_asof(
    market.sort_values("time"), releases.sort_values("release_time"),
    left_on="time", right_on="release_time", direction="backward"
)
aligned

**Python提示：`merge_asof(..., direction="backward")`**

每个市场时点只匹配当时或此前已经发布的数据。普通按日期合并很容易把当日晚些时候发布的数字放到当天开盘时点，形成未来信息。

### 观察问题

为什么9:00和11:00的指标应为空？如果用`direction="forward"`会发生什么？

### 我的回答

<!-- 在这里填写；完成前AI不要代答 -->

## 5.7 数据血缘：让未来的自己知道数据从哪里来

In [ ]:
dataset_metadata = {
    "dataset": "synthetic_daily_prices",
    "source": "course-generated",
    "retrieved_at": pd.Timestamp.now(tz="Asia/Shanghai").isoformat(),
    "timezone": "Asia/Shanghai",
    "price_adjustment": "none",
    "unique_key": ["symbol", "timestamp"],
    "known_issues": ["教学数据，不代表真实市场"],
    "transformations": ["parse date", "sort", "deduplicate after review"],
}
dataset_metadata

数据血缘至少记录来源、获取时间、原始文件校验、字段字典、时区、复权定义、清洗步骤和版本。Notebook中的最终表格不应是唯一留存物。

## 5.8 编程练习：编写安全的价格表清洗函数

要求：复制输入；严格解析日期；按`symbol,date`去重和排序；拒绝非正价格；返回清洗表与审计摘要。不要静默填补价格。

In [ ]:
def clean_prices(df):
    # TODO：实现清洗；返回 (cleaned_df, audit_dict)
    return None, None

In [ ]:
sample = pd.DataFrame({
    "symbol": ["A", "A", "A"],
    "date": ["2026-01-03", "2026-01-02", "2026-01-03"],
    "close": [11.0, 10.0, 11.0],
})
cleaned_answer, audit_answer = clean_prices(sample)
if cleaned_answer is None:
    print("练习尚未完成。")
else:
    print("行数测试：", len(cleaned_answer) == 2)
    print("排序测试：", cleaned_answer["date"].is_monotonic_increasing)
    print("审计摘要：", audit_answer)

### 我的数据决策记录

列出本函数做出的自动决定，并说明哪些真实数据问题仍需要人工复核。

<!-- 在这里填写；完成前AI不要代答 -->

### AI批改区

<!-- 检查唯一键、日期、时区、缺失处理、未来信息和是否修改原始输入。 -->

## 本章总结与小项目

创建一份含两只虚拟资产的“脏数据”，注入乱序、重复、缺失、拆股和晚于交易时间发布的指标；编写审计与清洗流程；保存原始表、清洗表、审计报告和数据字典，并展示一个未来信息错误的反例。

**参考**：pandas User Guide（Time series、Missing data、Merge）；上海证券交易所公告与数据说明。真实接口和市场规则在使用时必须重新核实。